In [1]:
import os
import re
import requests

# List of MapBiomas URLs
#urls = [
#    "https://storage.googleapis.com/mapbiomas-public/initiatives/bolivia/collection_2/lclu/coverage/bolivia_coverage_2023.tif",
#    "https://storage.googleapis.com/mapbiomas-public/initiatives/paraguay/collection_2/mapbiomas_paraguay_collection2_integration_v1-classification_2023.tif",
#    "https://storage.googleapis.com/mapbiomas-public/initiatives/argentina/collection-1/coverage/argentina_coverage_2022.tif",
#    "https://storage.googleapis.com/mapbiomas-public/initiatives/amazon/lulc/collection_6/integration/mapbiomas_collection60_integration_v1-classification_2023.tif",
#    "https://storage.googleapis.com/mapbiomas-public/initiatives/colombia/collection_2/lulc/mapbiomas_colombia_collection2_integration_v1/mapbiomas_colombia_collection2_integration_v1-classification_2023.tif",
#    "https://storage.googleapis.com/mapbiomas-public/initiatives/venezuela/collection_2/lulc/integration/mapbiomas_venezuela_collection2_integration_v1-classification_2023.tif",
#    "https://storage.googleapis.com/mapbiomas-public/initiatives/uruguay/collection_2/lulc/mapbiomas_uruguay_collection2_integration_v1-classification_2023.tif",
#    "https://storage.googleapis.com/mapbiomas-public/initiatives/chile/coverage/chile_coverage_2022.tif",
#    "https://storage.googleapis.com/mapbiomas-public/initiatives/peru/collection_3/LULC/peru_collection3_integration_v1-classification_2024.tif",
#    "https://storage.googleapis.com/mapbiomas-public/initiatives/brasil/collection_9/lclu/coverage/brasil_coverage_2023.tif"
#]

urls = [
    "https://storage.googleapis.com/mapbiomas-public/initiatives/argentina/collection-1/coverage/argentina_coverage_2022.tif",
    "https://storage.googleapis.com/mapbiomas-public/initiatives/chile/coverage/chile_coverage_2022.tif"
#    "https://storage.googleapis.com/mapbiomas-public/initiatives/argentina/collection-1/coverage/argentina_coverage_2022.tif",
#    "https://storage.googleapis.com/mapbiomas-public/initiatives/amazon/lulc/collection_6/integration/mapbiomas_collection60_integration_v1-classification_2023.tif",
#    "https://storage.googleapis.com/mapbiomas-public/initiatives/colombia/collection_2/lulc/mapbiomas_colombia_collection2_integration_v1/mapbiomas_colombia_collection2_integration_v1-classification_2023.tif",
#    "https://storage.googleapis.com/mapbiomas-public/initiatives/venezuela/collection_2/lulc/integration/mapbiomas_venezuela_collection2_integration_v1-classification_2023.tif",
#    "https://storage.googleapis.com/mapbiomas-public/initiatives/uruguay/collection_2/lulc/mapbiomas_uruguay_collection2_integration_v1-classification_2023.tif",
#    "https://storage.googleapis.com/mapbiomas-public/initiatives/chile/coverage/chile_coverage_2022.tif",
#    "https://storage.googleapis.com/mapbiomas-public/initiatives/peru/collection_3/LULC/peru_collection3_integration_v1-classification_2024.tif",
#    "https://storage.googleapis.com/mapbiomas-public/initiatives/brasil/collection_9/lclu/coverage/brasil_coverage_2023.tif"
]


# Output directory
output_dir = r"D:\WRI\MapBiomas Time Series\time series"
os.makedirs(output_dir, exist_ok=True)

# Loop through each URL
for url in urls:
    # Try to extract year and country
    match_year = re.search(r"(\d{4})(?=\.tif$)", url)
    match_country = re.search(r"initiatives/([^/]+)/", url)
    if not match_year or not match_country:
        print(f"[SKIP] Could not extract metadata from URL: {url}")
        continue

    year = match_year.group(1)
    country = match_country.group(1).lower()

    # Define output filename and path
    filename = f"mapbiomas_{country}_{year}.tif"
    output_path = os.path.join(output_dir, filename)

    # Skip if already downloaded
    if os.path.exists(output_path):
        print(f"[SKIP] Already exists: {filename}")
        continue

    # Attempt download
    try:
        print(f"[DOWNLOAD] {filename} from {url}")
        response = requests.get(url, stream=True, timeout=60)
        response.raise_for_status()
        with open(output_path, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        print(f"[SUCCESS] Downloaded: {filename}")
    except Exception as e:
        print(f"[ERROR] Failed to download {filename}: {e}")
        continue


[SKIP] Already exists: mapbiomas_argentina_2022.tif
[SKIP] Already exists: mapbiomas_chile_2022.tif


In [2]:
import os
import requests

# Output directory
output_dir = r"D:\WRI\MapBiomas Time Series\time series"
os.makedirs(output_dir, exist_ok=True)

# Countries with their URL pattern and filename template
countries = {
    "brasil":     ("brasil/collection_9/lclu/coverage/", "brasil_coverage_{year}.tif"),
    "bolivia":    ("bolivia/collection_2/lclu/coverage/", "bolivia_coverage_{year}.tif"),
    "paraguay":   ("paraguay/collection_2/", "mapbiomas_paraguay_collection2_integration_v1-classification_{year}.tif"),
    "argentina":  ("argentina/collection-1/coverage/", "argentina_coverage_{year}.tif"),
    "amazon":     ("amazon/lulc/collection_6/integration/", "mapbiomas_collection60_integration_v1-classification_{year}.tif"),
    "colombia":   ("colombia/collection_2/lulc/mapbiomas_colombia_collection2_integration_v1/", "mapbiomas_colombia_collection2_integration_v1-classification_{year}.tif"),
    "venezuela":  ("venezuela/collection_2/lulc/integration/", "mapbiomas_venezuela_collection2_integration_v1-classification_{year}.tif"),
    "uruguay":    ("uruguay/collection_2/lulc/", "mapbiomas_uruguay_collection2_integration_v1-classification_{year}.tif"),
    "chile":      ("chile/coverage/", "chile_coverage_{year}.tif"),
    "peru":       ("peru/collection_3/LULC/", "peru_collection3_integration_v1-classification_{year}.tif"),
}

# Base URL
base_url = "https://storage.googleapis.com/mapbiomas-public/initiatives"

# Loop through each country and year
for country, (path, template) in countries.items():
    for year in range(1985, 2024):  # 1985 to 2023 inclusive
        filename = template.format(year=year)
        full_url = f"{base_url}/{path}{filename}"
        local_filename = f"mapbiomas_{country}_{year}.tif"
        output_path = os.path.join(output_dir, local_filename)

        # Skip if already downloaded
        if os.path.exists(output_path):
            print(f"[SKIP] Already exists: {local_filename}")
            continue

        # Attempt download
        try:
            print(f"[TRY] {local_filename}")
            response = requests.get(full_url, stream=True, timeout=60)
            if response.status_code == 200:
                with open(output_path, 'wb') as f:
                    for chunk in response.iter_content(chunk_size=8192):
                        f.write(chunk)
                print(f"[SUCCESS] Downloaded: {local_filename}")
            else:
                print(f"[404] Not found: {local_filename}")
        except Exception as e:
            print(f"[ERROR] Failed {local_filename}: {e}")


[TRY] mapbiomas_brasil_1985.tif
[SUCCESS] Downloaded: mapbiomas_brasil_1985.tif
[TRY] mapbiomas_brasil_1986.tif
[SUCCESS] Downloaded: mapbiomas_brasil_1986.tif
[TRY] mapbiomas_brasil_1987.tif
[SUCCESS] Downloaded: mapbiomas_brasil_1987.tif
[TRY] mapbiomas_brasil_1988.tif
[SUCCESS] Downloaded: mapbiomas_brasil_1988.tif
[TRY] mapbiomas_brasil_1989.tif
[SUCCESS] Downloaded: mapbiomas_brasil_1989.tif
[TRY] mapbiomas_brasil_1990.tif
[SUCCESS] Downloaded: mapbiomas_brasil_1990.tif
[TRY] mapbiomas_brasil_1991.tif
[SUCCESS] Downloaded: mapbiomas_brasil_1991.tif
[TRY] mapbiomas_brasil_1992.tif
[SUCCESS] Downloaded: mapbiomas_brasil_1992.tif
[TRY] mapbiomas_brasil_1993.tif
[SUCCESS] Downloaded: mapbiomas_brasil_1993.tif
[TRY] mapbiomas_brasil_1994.tif
[SUCCESS] Downloaded: mapbiomas_brasil_1994.tif
[TRY] mapbiomas_brasil_1995.tif
[SUCCESS] Downloaded: mapbiomas_brasil_1995.tif
[TRY] mapbiomas_brasil_1996.tif
[SUCCESS] Downloaded: mapbiomas_brasil_1996.tif
[TRY] mapbiomas_brasil_1997.tif
[SUCCESS